# AutoResearch on Colab
基於 [karpathy/autoresearch](https://github.com/karpathy/autoresearch)

**使用方式：**
1. Runtime → Change runtime type → 選 **GPU (T4)**
2. 從上到下依序執行每個 cell
3. 訓練完成後結果會自動回傳到 MD.Piece 後端

**需求：** NVIDIA GPU (T4 免費即可)、CUDA driver

In [ ]:
# Step 1: 確認 GPU
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU! Go to Runtime → Change runtime type → GPU")

In [ ]:
# Step 2: 安裝 uv
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = os.path.expanduser('~/.local/bin') + ':' + os.environ['PATH']

In [ ]:
# Step 3: Clone autoresearch
!git clone https://github.com/karpathy/autoresearch.git
%cd autoresearch

In [ ]:
# Step 4: 安裝依賴
!uv sync

In [ ]:
# Step 5: 資料準備（下載 + tokenizer，約 2 分鐘）
!uv run prepare.py --num-shards 10

In [ ]:
# Step 6: 訓練（約 5 分鐘）
import time
import subprocess

start = time.time()
result = subprocess.run(
    ["uv", "run", "train.py"],
    capture_output=True, text=True
)
duration = time.time() - start

print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[-500:])

# 解析 val_bpb
import re
val_bpb = None
train_loss = None
steps = None

for line in result.stdout.split("\n"):
    m = re.search(r'val_bpb[=:\s]+([\d.]+)', line)
    if m:
        val_bpb = float(m.group(1))
    m = re.search(r'(?:train_)?loss[=:\s]+([\d.]+)', line)
    if m:
        train_loss = float(m.group(1))
    m = re.search(r'step[=:\s]+([\d]+)', line)
    if m:
        steps = int(m.group(1))

print(f"\n--- 結果 ---")
print(f"val_bpb: {val_bpb}")
print(f"train_loss: {train_loss}")
print(f"steps: {steps}")
print(f"duration: {duration:.0f}s")

In [ ]:
# Step 7: 回傳結果到 MD.Piece 後端
# ⚠️ 如果後端跑在本機，請用 ngrok 或改成你的公開 URL
MDPIECE_API = "http://localhost:8000"  # ← 改成你的後端 URL

import requests
import json

payload = {
    "name": "colab-t4-baseline",
    "val_bpb": val_bpb,
    "train_loss": train_loss,
    "steps": steps,
    "duration_seconds": duration,
    "notes": f"Colab T4 GPU, PyTorch {torch.__version__}",
    "model_config_summary": "default autoresearch config",
}

try:
    resp = requests.post(f"{MDPIECE_API}/research/", json=payload, timeout=10)
    print("✅ 結果已回傳:", resp.json())
except Exception as e:
    print(f"⚠️ 無法回傳到 MD.Piece 後端: {e}")
    print("你可以手動在前端 '自動研究' 頁面提交結果")
    print(f"\n手動提交資料:\n{json.dumps(payload, indent=2, ensure_ascii=False)}")